# Coopeval bootstrap interface discovery

This notebook belongs to the project's sequential measurement and validation programme. Read its result as evidence about behavioural validity, representation, comparator strength, timing, information matching, or mechanistic calibration as appropriate. Legacy H1/H_priv identifiers may remain inside code, saved paths, or frozen schemas for reproducibility; they are not the object being "found" by the current analysis.

**Repository framing.** The current paper separates target-specific representation from predictive privilege. A positive neural result is interpreted only after behavioural validity, control, comparator-strength, timing, and information-set checks.


# Latent reservations — Notebook 04
## CoopEval bootstrap + Traveler's Dilemma interface discovery

This notebook starts the independent replication environment **only after the primary LLM-Deliberation evidence has been frozen**.

Its job is deliberately narrow:

1. verify the Notebook 03 freeze;
2. clone and pin the official CoopEval repository;
3. install the repository in the active Python environment;
4. exercise the author's package/test/experiment entry points without modifying upstream source;
5. locate the exact Traveler's Dilemma and single-shot no-mechanism implementations;
6. discover the local-model agent path, prompt path, action representation, payoff path, and output schema;
7. write a machine-readable discovery manifest for the next replication notebook.

This is not the replication result itself. It is the equivalent of the earlier environment bootstrap: the next notebook will be built from the interfaces actually discovered here instead of guessing repository APIs.

**Output isolation:** all artifacts produced by this notebook live under:

`/workspace/latent-reservations/notebook_outputs/04_coopeval_bootstrap/<RUN_ID>/`

The shared upstream checkout lives under `vendor/coopeval/`.


In [1]:
NOTEBOOK_BUILD = "latent-reservations-notebook04-coopeval-bootstrap-interface-discovery-v2"
print("=" * 78)
print(f"NOTEBOOK BUILD: {NOTEBOOK_BUILD}")
print("CoopEval bootstrap + interface discovery")
print("=" * 78)


NOTEBOOK BUILD: latent-reservations-notebook04-coopeval-bootstrap-interface-discovery-v2
CoopEval bootstrap + interface discovery


## 1. Bootstrap dependencies

The official repository declares Python 3.12 and is installed editable after cloning. `pytest` is installed so repository tests can be exercised when present.


In [2]:
import importlib.util
import subprocess
import sys

bootstrap_packages = [
    "pyyaml>=6",
    "tqdm>=4.66",
    "pytest>=8,<9",
]
subprocess.run(
    [sys.executable, "-m", "pip", "install", *bootstrap_packages],
    check=True,
)
print("Notebook 04 bootstrap dependencies ready.")


Notebook 04 bootstrap dependencies ready.


## 2. Paths and isolated run directory


In [3]:
from __future__ import annotations

import hashlib
import importlib
import inspect
import json
import os
import pkgutil
import queue
import re
import shlex
import shutil
import subprocess
import sys
import threading
import time
from datetime import datetime, timezone
from pathlib import Path

import yaml
from tqdm.auto import tqdm

PROJECT_ROOT = Path(os.environ.get("LR_PROJECT_ROOT", "/workspace/latent-reservations")).expanduser().resolve()
NOTEBOOK_SLUG = "04_coopeval_bootstrap"
RUN_ID = os.environ.get("LR_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

NOTEBOOK_OUTPUT_BASE = PROJECT_ROOT / "notebook_outputs" / NOTEBOOK_SLUG
RUN_OUTPUT_DIR = NOTEBOOK_OUTPUT_BASE / RUN_ID
MANIFEST_DIR = RUN_OUTPUT_DIR / "manifests"
DATA_DIR = RUN_OUTPUT_DIR / "data"
RESULTS_DIR = RUN_OUTPUT_DIR / "results"
TABLE_DIR = RESULTS_DIR / "tables"
LOG_DIR = RUN_OUTPUT_DIR / "logs"

UPSTREAM_DIR = PROJECT_ROOT / "vendor" / "coopeval"
COOPEVAL_URL = "https://github.com/Xiao215/CoopEval.git"

for p in [
    MANIFEST_DIR,
    DATA_DIR / "source_snapshots",
    TABLE_DIR,
    RESULTS_DIR / "figures",
    LOG_DIR,
    PROJECT_ROOT / "vendor",
]:
    p.mkdir(parents=True, exist_ok=True)

NOTEBOOK_OUTPUT_BASE.mkdir(parents=True, exist_ok=True)
latest_pointer = {
    "notebook_slug": NOTEBOOK_SLUG,
    "run_id": RUN_ID,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "updated_at_utc": datetime.now(timezone.utc).isoformat(),
}
(NOTEBOOK_OUTPUT_BASE / "latest_run.json").write_text(json.dumps(latest_pointer, indent=2))

print(f"PROJECT_ROOT   = {PROJECT_ROOT}")
print(f"RUN_OUTPUT_DIR = {RUN_OUTPUT_DIR}")
print(f"UPSTREAM_DIR   = {UPSTREAM_DIR}")


PROJECT_ROOT   = /workspace/latent-reservations
RUN_OUTPUT_DIR = /workspace/latent-reservations/notebook_outputs/04_coopeval_bootstrap/20260814T064506Z
UPSTREAM_DIR   = /workspace/latent-reservations/vendor/coopeval


## 3. Require the frozen Notebook 03 evidence package

Notebook 04 does not alter any LLM-Deliberation artifacts. It records the frozen result as provenance for why an independent replication is being attempted.


In [4]:
NB03_BASE = PROJECT_ROOT / "notebook_outputs" / "03_llm_deliberation_evidence_freeze"
nb03_latest_path = NB03_BASE / "latest_run.json"
if not nb03_latest_path.exists():
    raise FileNotFoundError(f"Notebook 03 latest-run pointer not found: {nb03_latest_path}")

nb03_latest = json.loads(nb03_latest_path.read_text())
NB03_RUN_DIR = Path(nb03_latest["run_output_dir"])
nb03_freeze_path = NB03_RUN_DIR / "manifests" / "notebook03_freeze_manifest.json"
nb03_summary_path = NB03_RUN_DIR / "results" / "tables" / "frozen_evidence_summary.json"

for p in [nb03_freeze_path, nb03_summary_path]:
    if not p.exists():
        raise FileNotFoundError(p)

nb03_freeze = json.loads(nb03_freeze_path.read_text())
nb03_summary = json.loads(nb03_summary_path.read_text())
if nb03_freeze.get("frozen") is not True:
    raise RuntimeError("Notebook 03 exists but is not marked frozen.")

provenance = {
    "notebook03_run": str(NB03_RUN_DIR),
    "notebook03_frozen": True,
    "h1_delta_latent": nb03_summary["h1_primary"]["delta_latent_auditor_minus_activation"],
    "h1_ci95": nb03_summary["h1_primary"]["cluster_bootstrap_95ci"],
    "h3_gap": nb03_summary["h3_reports"]["public_minus_private_error_gap"],
}
(MANIFEST_DIR / "prior_evidence.json").write_text(json.dumps(provenance, indent=2))
print(json.dumps(provenance, indent=2))


{
  "notebook03_run": "/workspace/latent-reservations/notebook_outputs/03_llm_deliberation_evidence_freeze/20260814T014816Z",
  "notebook03_frozen": true,
  "h1_delta_latent": -14.495083109537761,
  "h1_ci95": [
    -25.559805552164715,
    -7.303814188639323
  ],
  "h3_gap": 0.0
}


## 4. Clone or reuse the official CoopEval checkout

The checkout is shared across notebooks; experimental outputs are not. Existing untracked files are allowed, but tracked source changes are recorded and block scientific use.


In [5]:
if not UPSTREAM_DIR.exists():
    print(f"Cloning {COOPEVAL_URL} -> {UPSTREAM_DIR}")
    subprocess.run(["git", "clone", COOPEVAL_URL, str(UPSTREAM_DIR)], check=True)
else:
    print(f"Reusing existing checkout: {UPSTREAM_DIR}")

actual_url = subprocess.check_output(
    ["git", "remote", "get-url", "origin"],
    cwd=UPSTREAM_DIR,
    text=True,
).strip()
if "Xiao215/CoopEval" not in actual_url:
    raise RuntimeError(f"Unexpected CoopEval origin: {actual_url}")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=UPSTREAM_DIR, text=True).strip()
branch = subprocess.check_output(
    ["git", "branch", "--show-current"], cwd=UPSTREAM_DIR, text=True
).strip()

tracked_status = subprocess.check_output(
    ["git", "status", "--porcelain", "--untracked-files=no"],
    cwd=UPSTREAM_DIR,
    text=True,
).strip()
untracked_status = subprocess.check_output(
    ["git", "ls-files", "--others", "--exclude-standard"],
    cwd=UPSTREAM_DIR,
    text=True,
).splitlines()

worktree_record = {
    "url": actual_url,
    "commit": commit,
    "branch": branch,
    "tracked_status": tracked_status,
    "untracked_files": untracked_status,
}
(MANIFEST_DIR / "coopeval_worktree.json").write_text(json.dumps(worktree_record, indent=2))

if tracked_status:
    raise RuntimeError(
        "The official CoopEval checkout has tracked modifications. "
        "Leave upstream source unchanged and put project-specific logic outside vendor/coopeval.\n"
        + tracked_status
    )

print(json.dumps({
    "url": actual_url,
    "commit": commit,
    "branch": branch,
    "untracked_files": len(untracked_status),
}, indent=2))


Reusing existing checkout: /workspace/latent-reservations/vendor/coopeval
{
  "url": "https://github.com/Xiao215/CoopEval.git",
  "commit": "8559e858b5cc33e0a29ac592c0b21eab007feb57",
  "branch": "main",
  "untracked_files": 0
}


## 5. Install the pinned repository editable


In [6]:
install_cmd = [sys.executable, "-m", "pip", "install", "-e", str(UPSTREAM_DIR)]
print(" ".join(shlex.quote(x) for x in install_cmd))
subprocess.run(install_cmd, check=True)

# Verify the editable install in a fresh interpreter. Editable installs commonly
# rely on a .pth file that is processed at interpreter startup, so this is the
# authoritative install check.
fresh_import = subprocess.run(
    [
        sys.executable,
        "-c",
        "import coopeval; print(coopeval.__file__)",
    ],
    cwd=UPSTREAM_DIR,
    text=True,
    capture_output=True,
)
if fresh_import.returncode != 0:
    raise RuntimeError(
        "CoopEval editable install completed but a fresh Python process cannot import coopeval.\n"
        + fresh_import.stderr[-12000:]
    )

# Make the same src-layout package visible to the *current* Jupyter kernel
# without requiring a kernel restart. A running interpreter does not re-process
# newly-created editable-install .pth files automatically.
COOPEVAL_SRC_DIR = (UPSTREAM_DIR / "src").resolve()
if not (COOPEVAL_SRC_DIR / "coopeval").is_dir():
    raise FileNotFoundError(f"Expected package directory not found: {COOPEVAL_SRC_DIR / 'coopeval'}")

src_text = str(COOPEVAL_SRC_DIR)
if src_text not in sys.path:
    sys.path.insert(0, src_text)
importlib.invalidate_caches()

spec = importlib.util.find_spec("coopeval")
if spec is None:
    raise ModuleNotFoundError(
        f"coopeval is still unavailable in the active kernel after adding {COOPEVAL_SRC_DIR} to sys.path"
    )

print("Editable CoopEval install complete.")
print(f"Fresh-process import: {fresh_import.stdout.strip()}")
print(f"Current-kernel package spec: {spec.origin}")


/usr/local/bin/python -m pip install -e /workspace/latent-reservations/vendor/coopeval
Obtaining file:///workspace/latent-reservations/vendor/coopeval
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Building editable for coopeval (pyproject.toml): started
  Building editable for coopeval (pyproject.toml): finished with status 'done'
  Created wheel for coopeval: filename=coopeval-2.0.0-0.editable-py3-none-any.whl size=1382 sha256=6860fc8e67e55b26a65234cf92fabe985bf0ee6484deb64b00341a719ab3426a
  Stored in directory: /tmp/pip-ephem-wheel-cache-i

## 6. Capture environment, repository shape, and license


In [7]:
import platform

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

try:
    import torch
    torch_record = {
        "torch": torch.__version__,
        "torch_cuda": torch.version.cuda,
        "cuda_available": torch.cuda.is_available(),
        "cuda_device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }
except Exception as exc:
    torch_record = {"error": repr(exc)}

license_candidates = sorted(
    [p for p in UPSTREAM_DIR.iterdir() if p.is_file() and p.name.lower().startswith(("license", "copying"))]
)
license_record = []
for p in license_candidates:
    license_record.append({
        "path": str(p.relative_to(UPSTREAM_DIR)),
        "sha256": sha256_file(p),
        "first_lines": p.read_text(errors="replace").splitlines()[:8],
    })

shape = {}
for rel in [
    "README.md",
    "pyproject.toml",
    "configs",
    "src/coopeval",
    "src/coopeval/games",
    "src/coopeval/mechanisms",
    "src/coopeval/agents",
    "scripts/experiments",
]:
    shape[rel] = (UPSTREAM_DIR / rel).exists()

environment_record = {
    "python": platform.python_version(),
    "executable": sys.executable,
    **torch_record,
    "repo_commit": commit,
    "repo_branch": branch,
    "repo_shape": shape,
    "license_files": license_record,
}
(MANIFEST_DIR / "environment_and_repo.json").write_text(json.dumps(environment_record, indent=2))
print(json.dumps(environment_record, indent=2))


{
  "python": "3.12.3",
  "executable": "/usr/local/bin/python",
  "torch": "2.8.0+cu128",
  "torch_cuda": "12.8",
  "cuda_available": true,
  "cuda_device": "NVIDIA L40S",
  "repo_commit": "8559e858b5cc33e0a29ac592c0b21eab007feb57",
  "repo_branch": "main",
  "repo_shape": {
    "README.md": true,
    "pyproject.toml": true,
    "configs": true,
    "src/coopeval": true,
    "src/coopeval/games": true,
    "src/coopeval/mechanisms": true,
    "src/coopeval/agents": true,
    "scripts/experiments": true
  },
  "license_files": []
}


## 7. Streamed subprocess helper

Long subprocesses print live output. If a process is quiet, a heartbeat reports elapsed time and log size so there are no silent multi-minute sections.


In [8]:
def run_with_heartbeat(
    cmd,
    *,
    cwd: Path,
    log_path: Path,
    timeout_s: int,
    heartbeat_s: int = 30,
    env=None,
):
    cmd = [str(x) for x in cmd]
    print("$ " + " ".join(shlex.quote(x) for x in cmd))
    print(f"log: {log_path}")
    started = time.time()

    proc = subprocess.Popen(
        cmd,
        cwd=str(cwd),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    q = queue.Queue()
    sentinel = object()

    def reader():
        try:
            for line in proc.stdout:
                q.put(line)
        finally:
            q.put(sentinel)

    thread = threading.Thread(target=reader, daemon=True)
    thread.start()

    last_heartbeat = time.time()
    lines = []
    with log_path.open("w") as log:
        stream_done = False
        while True:
            now = time.time()
            if now - started > timeout_s:
                proc.kill()
                raise TimeoutError(
                    f"Process exceeded {timeout_s}s. See {log_path}"
                )

            try:
                item = q.get(timeout=1.0)
                if item is sentinel:
                    stream_done = True
                else:
                    lines.append(item)
                    log.write(item)
                    log.flush()
                    print(item, end="")
            except queue.Empty:
                pass

            if now - last_heartbeat >= heartbeat_s and proc.poll() is None:
                elapsed_min = (now - started) / 60
                kib = log_path.stat().st_size / 1024 if log_path.exists() else 0
                print(f"[heartbeat] {elapsed_min:.1f} min elapsed | log={kib:.1f} KiB")
                last_heartbeat = now

            if proc.poll() is not None and stream_done and q.empty():
                break

    return {
        "cmd": cmd,
        "returncode": int(proc.returncode),
        "elapsed_s": round(time.time() - started, 3),
        "log": str(log_path),
        "output_tail": "".join(lines)[-16000:],
    }


## 8. Exercise the official experiment entry point

This verifies that the installed package and the repository's own experiment runner can start in the current environment.


In [9]:
runner_candidates = sorted((UPSTREAM_DIR / "scripts" / "experiments").glob("run_experiment.py"))
if not runner_candidates:
    raise FileNotFoundError("Official scripts/experiments/run_experiment.py was not found.")

RUNNER = runner_candidates[0]
runner_help_record = run_with_heartbeat(
    [sys.executable, str(RUNNER), "--help"],
    cwd=UPSTREAM_DIR,
    log_path=LOG_DIR / "run_experiment_help.log",
    timeout_s=300,
)
(MANIFEST_DIR / "native_runner_help.json").write_text(json.dumps(runner_help_record, indent=2))

if runner_help_record["returncode"] != 0:
    raise RuntimeError(
        "The official CoopEval experiment entry point failed to start.\n"
        + runner_help_record["output_tail"]
    )

print(f"Official experiment runner starts successfully ({runner_help_record['elapsed_s']}s).")


$ /usr/local/bin/python /workspace/latent-reservations/vendor/coopeval/scripts/experiments/run_experiment.py --help
log: /workspace/latent-reservations/notebook_outputs/04_coopeval_bootstrap/20260814T064506Z/logs/run_experiment_help.log
usage: run_experiment.py [-h] --config CONFIG [--wandb]
                         [--matchup-payoffs MATCHUP_PAYOFFS]
                         [--output-dir OUTPUT_DIR]
                         [--experiment-name EXPERIMENT_NAME] [--seed SEED]

Run game-theoretic experiments with configurable evaluations

options:
  -h, --help            show this help message and exit
  --config CONFIG       Config YAML file name
  --wandb               Enable Weights & Biases figure saving
  --matchup-payoffs MATCHUP_PAYOFFS
                        Path to a JSON file containing precomputed matchup
                        payoffs.
  --output-dir OUTPUT_DIR
                        Custom output directory for this experiment (overrides
                        default tim

## 9. Locate the intended game/mechanism/config files

The project plan starts with Traveler's Dilemma and the single-shot no-mechanism condition. Paths are discovered from the checkout rather than hard-coded beyond the official top-level layout.


In [10]:
def normalized_name(path: Path) -> str:
    return re.sub(r"[^a-z0-9]+", "", path.stem.lower())

game_yamls = sorted((UPSTREAM_DIR / "configs" / "games").rglob("*.yaml"))
mechanism_yamls = sorted((UPSTREAM_DIR / "configs" / "mechanisms").rglob("*.yaml"))
agent_yamls = sorted((UPSTREAM_DIR / "configs" / "agents").rglob("*.yaml"))
evaluation_yamls = sorted((UPSTREAM_DIR / "configs" / "evaluation").rglob("*.yaml"))

traveler_candidates = [
    p for p in game_yamls
    if "travel" in normalized_name(p)
]
no_mechanism_candidates = [
    p for p in mechanism_yamls
    if "nomechanism" in normalized_name(p) or normalized_name(p) in {"none", "baseline"}
]

if not traveler_candidates:
    raise FileNotFoundError(
        f"No Traveler's Dilemma YAML found under {UPSTREAM_DIR / 'configs/games'}"
    )
if not no_mechanism_candidates:
    raise FileNotFoundError(
        f"No no-mechanism YAML found under {UPSTREAM_DIR / 'configs/mechanisms'}"
    )

TRAVELER_CONFIG = traveler_candidates[0]
NO_MECHANISM_CONFIG = no_mechanism_candidates[0]

config_inventory = {
    "traveler_game_config": str(TRAVELER_CONFIG.relative_to(UPSTREAM_DIR)),
    "no_mechanism_config": str(NO_MECHANISM_CONFIG.relative_to(UPSTREAM_DIR)),
    "agent_configs": [str(p.relative_to(UPSTREAM_DIR)) for p in agent_yamls],
    "evaluation_configs": [str(p.relative_to(UPSTREAM_DIR)) for p in evaluation_yamls],
    "traveler_config_data": yaml.safe_load(TRAVELER_CONFIG.read_text()),
    "no_mechanism_config_data": yaml.safe_load(NO_MECHANISM_CONFIG.read_text()),
}
(TABLE_DIR / "official_config_inventory.json").write_text(json.dumps(config_inventory, indent=2, default=str))

print(json.dumps({
    "traveler_game_config": config_inventory["traveler_game_config"],
    "no_mechanism_config": config_inventory["no_mechanism_config"],
    "agent_config_count": len(agent_yamls),
    "evaluation_config_count": len(evaluation_yamls),
}, indent=2))
print("\nTraveler config:")
print(TRAVELER_CONFIG.read_text())
print("\nNo-mechanism config:")
print(NO_MECHANISM_CONFIG.read_text())


{
  "traveler_game_config": "configs/games/travellers_dilemma.yaml",
  "no_mechanism_config": "configs/mechanisms/no_mechanism.yaml",
  "agent_config_count": 5,
  "evaluation_config_count": 2
}

Traveler config:
type: TravellersDilemma
kwargs:
  min_claim: 2
  num_actions: 4
  claim_spacing: 1
  bonus: 2.0


No-mechanism config:
type: NoMechanism
kwargs: {}



## 10. Import and discover the official game/mechanism classes

All modules below `coopeval.games` and `coopeval.mechanisms` are imported. Candidate classes, constructors, public methods, and source paths are recorded for the adapter notebook.


In [11]:
# Defensive src-layout path injection for an already-running Jupyter kernel.
COOPEVAL_SRC_DIR = (UPSTREAM_DIR / "src").resolve()
if str(COOPEVAL_SRC_DIR) not in sys.path:
    sys.path.insert(0, str(COOPEVAL_SRC_DIR))
importlib.invalidate_caches()

import coopeval
import coopeval.games
import coopeval.mechanisms

def discover_classes(package, predicate):
    rows = []
    for modinfo in tqdm(
        list(pkgutil.walk_packages(package.__path__, package.__name__ + ".")),
        desc=f"Import {package.__name__}",
        unit="module",
    ):
        try:
            module = importlib.import_module(modinfo.name)
        except Exception as exc:
            rows.append({
                "module": modinfo.name,
                "import_error": repr(exc),
            })
            continue

        for name, obj in vars(module).items():
            if not inspect.isclass(obj):
                continue
            if obj.__module__ != module.__name__:
                continue
            if not predicate(name):
                continue
            try:
                source_file = inspect.getsourcefile(obj)
            except Exception:
                source_file = None
            methods = []
            for method_name, method in inspect.getmembers(obj, predicate=inspect.isfunction):
                if method_name.startswith("_"):
                    continue
                try:
                    sig = str(inspect.signature(method))
                except Exception:
                    sig = "<unavailable>"
                methods.append({"name": method_name, "signature": sig})
            try:
                constructor = str(inspect.signature(obj))
            except Exception:
                constructor = "<unavailable>"
            rows.append({
                "module": module.__name__,
                "class": name,
                "constructor": constructor,
                "source_file": source_file,
                "public_methods": methods,
            })
    return rows

traveler_classes = discover_classes(
    coopeval.games,
    lambda name: "travel" in name.lower() and "dilemma" in name.lower(),
)
no_mechanism_classes = discover_classes(
    coopeval.mechanisms,
    lambda name: "mechanism" in name.lower() and ("no" in name.lower() or "none" in name.lower()),
)

if not any("class" in row for row in traveler_classes):
    raise RuntimeError("No Traveler's Dilemma class could be imported from coopeval.games.")
if not any("class" in row for row in no_mechanism_classes):
    raise RuntimeError("No no-mechanism class could be imported from coopeval.mechanisms.")

class_discovery = {
    "traveler_classes": traveler_classes,
    "no_mechanism_classes": no_mechanism_classes,
}
(TABLE_DIR / "class_discovery.json").write_text(json.dumps(class_discovery, indent=2, default=str))
print(json.dumps(class_discovery, indent=2, default=str))


Import coopeval.games:   0%|          | 0/7 [00:00<?, ?module/s]

Import coopeval.mechanisms:   0%|          | 0/8 [00:00<?, ?module/s]

{
  "traveler_classes": [
    {
      "module": "coopeval.games.travellers_dilemma",
      "class": "TravellersDilemma",
      "constructor": "(*, min_claim: 'int', num_actions: 'int', claim_spacing: 'int', bonus: 'float') -> 'None'",
      "source_file": "/workspace/latent-reservations/vendor/coopeval/src/coopeval/games/travellers_dilemma.py",
      "public_methods": [
        {
          "name": "add_mediator_action",
          "signature": "(self) -> None"
        },
        {
          "name": "get_action_self_payoff",
          "signature": "(self, action: 'Action') -> 'float'"
        },
        {
          "name": "get_actions_payoff",
          "signature": "(self, actions: 'Sequence[Action]') -> 'Sequence[float]'"
        },
        {
          "name": "get_player_prompt",
          "signature": "(self, player_id: int) -> str"
        },
        {
          "name": "play",
          "signature": "(self, additional_info: 'list[str] | str', players: 'Sequence[Agent]', action_map

## 11. Exercise repository-native tests for the target game when available

The notebook searches for tests mentioning the target game, no-mechanism baseline, or generic game behavior. It runs the narrowest available test set and captures live output. If the repository ships no matching tests, that fact is recorded rather than inventing a test.


In [12]:
test_roots = [
    UPSTREAM_DIR / "tests",
    UPSTREAM_DIR / "scripts" / "tests",
]
all_test_files = []
for root in test_roots:
    if root.exists():
        all_test_files.extend(sorted(root.rglob("test*.py")))

target_tests = []
generic_tests = []
for p in all_test_files:
    text = p.read_text(errors="replace").lower()
    rel = str(p.relative_to(UPSTREAM_DIR)).lower()
    if "travel" in text or "travel" in rel or "no_mechanism" in text or "nomechanism" in text:
        target_tests.append(p)
    elif "game" in text or "mechanism" in text:
        generic_tests.append(p)

selected_tests = target_tests if target_tests else generic_tests[:5]

test_record = {
    "available_test_files": [str(p.relative_to(UPSTREAM_DIR)) for p in all_test_files],
    "selected_tests": [str(p.relative_to(UPSTREAM_DIR)) for p in selected_tests],
    "attempted": bool(selected_tests),
}

if selected_tests:
    cmd = [sys.executable, "-m", "pytest", "-q"] + [
        str(p.relative_to(UPSTREAM_DIR)) for p in selected_tests
    ]
    native_test_run = run_with_heartbeat(
        cmd,
        cwd=UPSTREAM_DIR,
        log_path=LOG_DIR / "native_target_tests.log",
        timeout_s=1800,
    )
    test_record["run"] = native_test_run
    if native_test_run["returncode"] != 0:
        print(
            "Targeted repository tests did not all pass. "
            "The failure is retained for adapter diagnosis rather than hidden."
        )
else:
    print("No repository-native matching tests were found.")

(MANIFEST_DIR / "native_target_tests.json").write_text(json.dumps(test_record, indent=2))
print(json.dumps({
    "attempted": test_record["attempted"],
    "selected_tests": test_record["selected_tests"],
    "returncode": test_record.get("run", {}).get("returncode"),
}, indent=2))


No repository-native matching tests were found.
{
  "attempted": false,
  "selected_tests": [],
  "returncode": null
}


## 12. Locate prompt construction, action parsing, payoff logic, and local-model support

This is source discovery, not a source modification. Short matching snippets are written to machine-readable artifacts so the next notebook can target the actual interfaces.


In [13]:
SCAN_SUFFIXES = {".py", ".yaml", ".yml", ".toml"}
scan_files = [
    p for p in UPSTREAM_DIR.rglob("*")
    if p.is_file()
    and p.suffix.lower() in SCAN_SUFFIXES
    and ".git" not in p.parts
]

patterns = {
    "prompt": re.compile(r"prompt|instruction|message", re.IGNORECASE),
    "action_parse": re.compile(r"parse.*action|action.*parse|extract.*action|valid.*action", re.IGNORECASE),
    "payoff": re.compile(r"payoff|utility|reward", re.IGNORECASE),
    "local_model": re.compile(
        r"hugging\s*face|huggingface|transformers|AutoModel|MODEL_WEIGHTS_DIR|local[_ -]?model|\bhf\b",
        re.IGNORECASE,
    ),
    "traveler": re.compile(r"travell?er'?s?\s*dilemma|travellers?dilemma|travelers?dilemma", re.IGNORECASE),
}

def scan_pattern(pattern, max_hits=80):
    hits = []
    for p in tqdm(scan_files, desc=f"Scan {pattern.pattern[:18]}", unit="file", leave=False):
        try:
            lines = p.read_text(errors="replace").splitlines()
        except Exception:
            continue
        for lineno, line in enumerate(lines, start=1):
            if pattern.search(line):
                start = max(0, lineno - 3)
                end = min(len(lines), lineno + 2)
                hits.append({
                    "path": str(p.relative_to(UPSTREAM_DIR)),
                    "line": lineno,
                    "snippet": "\n".join(
                        f"{i+1}: {lines[i]}" for i in range(start, end)
                    ),
                })
                if len(hits) >= max_hits:
                    return hits
    return hits

source_discovery = {
    key: scan_pattern(pattern)
    for key, pattern in patterns.items()
}
(TABLE_DIR / "source_interface_discovery.json").write_text(
    json.dumps(source_discovery, indent=2)
)

print(json.dumps({
    key: {
        "hits": len(value),
        "first_paths": list(dict.fromkeys(v["path"] for v in value))[:8],
    }
    for key, value in source_discovery.items()
}, indent=2))


Scan prompt|instruction:   0%|          | 0/279 [00:00<?, ?file/s]

Scan parse.*action|acti:   0%|          | 0/279 [00:00<?, ?file/s]

Scan payoff|utility|rew:   0%|          | 0/279 [00:00<?, ?file/s]

Scan hugging\s*face|hug:   0%|          | 0/279 [00:00<?, ?file/s]

Scan travell?er'?s?\s*d:   0%|          | 0/279 [00:00<?, ?file/s]

{
  "prompt": {
    "hits": 80,
    "first_paths": [
      "src/coopeval/visualization/analysis_utils.py",
      "src/coopeval/script_utils/display_helper.py",
      "src/coopeval/mechanisms/reputation.py",
      "src/coopeval/mechanisms/repetition.py",
      "src/coopeval/mechanisms/prompts.py",
      "src/coopeval/mechanisms/mediation.py",
      "src/coopeval/mechanisms/contracting.py"
    ]
  },
  "action_parse": {
    "hits": 17,
    "first_paths": [
      "src/coopeval/mechanisms/mediation.py",
      "src/coopeval/mechanisms/json_parsing.py",
      "src/coopeval/mechanisms/contracting.py",
      "src/coopeval/games/base.py",
      "scripts/analysis/plot_conditional_action_frequency.py",
      "scripts/analysis/plot_action_frequency.py"
    ]
  },
  "payoff": {
    "hits": 80,
    "first_paths": [
      "src/coopeval/logger_manager.py",
      "src/coopeval/visualization/analysis_utils.py",
      "src/coopeval/script_utils/display_helper.py",
      "src/coopeval/script_utils/colors.

## 13. Inspect shipped agent configs and prioritize a local-model path

No API-backed experiment is launched automatically. The next notebook should use the same local open-weight subject if the official code exposes a compatible local-model adapter.


In [14]:
agent_inventory = []
for p in tqdm(agent_yamls, desc="Inspect agent configs", unit="config"):
    text = p.read_text(errors="replace")
    data = yaml.safe_load(text)
    lower = text.lower()
    local_score = sum(
        token in lower
        for token in ["huggingface", "hugging face", "transformers", "local", "qwen", "hf_"]
    )
    agent_inventory.append({
        "path": str(p.relative_to(UPSTREAM_DIR)),
        "local_score": int(local_score),
        "data": data,
    })

agent_inventory.sort(key=lambda x: (-x["local_score"], x["path"]))
(TABLE_DIR / "agent_config_inventory.json").write_text(
    json.dumps(agent_inventory, indent=2, default=str)
)

print("Top agent configs by local-model evidence:")
for row in agent_inventory[:10]:
    print(f"  score={row['local_score']}  {row['path']}")

local_source_hits = source_discovery["local_model"]
print(f"\nLocal-model source hits: {len(local_source_hits)}")
for hit in local_source_hits[:12]:
    print(f"\n{hit['path']}:{hit['line']}\n{hit['snippet']}")


Inspect agent configs:   0%|          | 0/5 [00:00<?, ?config/s]

Top agent configs by local-model evidence:
  score=1  configs/agents/cheap_llms_3.yaml
  score=1  configs/agents/few_strong_llms.yaml
  score=1  configs/agents/sota_llms.yaml
  score=0  configs/agents/test_agents_3.yaml
  score=0  configs/agents/test_agents_6.yaml

Local-model source hits: 2

pyproject.toml:27
25:     "torch",
26:     "tqdm",
27:     "transformers",
28: ]
29: 

src/coopeval/config.py:12
10: FIGURE_DIR = PROJECT_ROOT / "figures"
11: CONFIG_DIR = PROJECT_ROOT / "configs"
12: MODEL_WEIGHTS_DIR = PROJECT_ROOT / "model-weights"
13: CACHE_DIR = PROJECT_ROOT / "caches"
14: DATA_DIR = PROJECT_ROOT / "data"


## 14. Inspect shipped output examples and summarize schemas

Existing author outputs are useful for locating trace/action fields without launching a large experiment.


In [15]:
output_root = UPSTREAM_DIR / "outputs"
output_examples = []

if output_root.exists():
    candidates = [
        p for p in output_root.rglob("*")
        if p.is_file() and p.suffix.lower() in {".json", ".jsonl", ".csv"}
    ]
else:
    candidates = []

for p in tqdm(candidates[:100], desc="Inspect output examples", unit="file"):
    rel = str(p.relative_to(UPSTREAM_DIR))
    try:
        if p.suffix.lower() == ".json":
            obj = json.loads(p.read_text(errors="replace"))
            if isinstance(obj, dict):
                schema = {"kind": "json_dict", "keys": sorted(obj.keys())[:80]}
            elif isinstance(obj, list):
                first = obj[0] if obj else None
                schema = {
                    "kind": "json_list",
                    "length": len(obj),
                    "first_keys": sorted(first.keys())[:80] if isinstance(first, dict) else None,
                }
            else:
                schema = {"kind": type(obj).__name__}
        elif p.suffix.lower() == ".jsonl":
            first_line = next((x for x in p.read_text(errors="replace").splitlines() if x.strip()), "")
            obj = json.loads(first_line) if first_line else None
            schema = {
                "kind": "jsonl",
                "first_keys": sorted(obj.keys())[:80] if isinstance(obj, dict) else None,
            }
        else:
            first_line = p.read_text(errors="replace").splitlines()[:1]
            schema = {"kind": "csv", "header": first_line[0] if first_line else ""}
        output_examples.append({"path": rel, "schema": schema})
    except Exception as exc:
        output_examples.append({"path": rel, "error": repr(exc)})

(TABLE_DIR / "author_output_schema_inventory.json").write_text(
    json.dumps(output_examples, indent=2)
)
print(f"Inspected {len(output_examples)} author output examples.")
for row in output_examples[:20]:
    print(row)


Inspect output examples:   0%|          | 0/11 [00:00<?, ?file/s]

Inspected 11 author output examples.
{'path': 'outputs/judge/full_gpt5.2/raw/judgement_summary.json', 'schema': {'kind': 'json_dict', 'keys': ['agent_type_filter', 'discovered_runs', 'dry_run', 'game_counts', 'generated_at_utc', 'judge_package_path', 'label_counts', 'max_items', 'max_workers', 'mean_confidence', 'mechanism_counts', 'min_response_chars', 'model_counts', 'output_file', 'processing_counters', 'rows', 'script', 'text_mode', 'unique_trace_ids']}}
{'path': 'outputs/judge/full_gpt5.2/raw/judgement.jsonl', 'schema': {'kind': 'jsonl', 'first_keys': ['action', 'agent_type', 'batch_name', 'classification_confidence', 'classification_explanation', 'classification_justification', 'classification_labels', 'dry_run', 'game', 'judge_input_chars', 'judge_model', 'judge_provider', 'mechanism', 'mediated', 'model', 'player', 'player_id', 'points', 'processed_at_utc', 'record_line', 'response_chars', 'response_source', 'run_dir', 'run_name', 'taxonomy_path', 'trace_id']}}
{'path': 'output

## 15. Write the adapter handoff manifest

The next notebook should consume this manifest rather than repeat broad repository discovery.


In [16]:
def first_class(rows):
    return next((row for row in rows if "class" in row), None)

traveler_class = first_class(traveler_classes)
no_mechanism_class = first_class(no_mechanism_classes)

handoff = {
    "notebook_build": NOTEBOOK_BUILD,
    "run_id": RUN_ID,
    "run_output_dir": str(RUN_OUTPUT_DIR),
    "prior_frozen_evidence": provenance,
    "coopeval": {
        "url": actual_url,
        "commit": commit,
        "branch": branch,
        "upstream_dir": str(UPSTREAM_DIR),
        "tracked_clean": tracked_status == "",
        "editable_install_passed": True,
        "runner_help_passed": runner_help_record["returncode"] == 0,
        "targeted_native_tests_attempted": test_record["attempted"],
        "targeted_native_tests_returncode": test_record.get("run", {}).get("returncode"),
    },
    "target": {
        "game": "Traveler's Dilemma",
        "game_config": str(TRAVELER_CONFIG.relative_to(UPSTREAM_DIR)),
        "game_class": traveler_class,
        "mechanism": "No Mechanism",
        "mechanism_config": str(NO_MECHANISM_CONFIG.relative_to(UPSTREAM_DIR)),
        "mechanism_class": no_mechanism_class,
        "behavioral_target": "continuous claim",
    },
    "discovery_artifacts": {
        "config_inventory": str(TABLE_DIR / "official_config_inventory.json"),
        "class_discovery": str(TABLE_DIR / "class_discovery.json"),
        "source_interfaces": str(TABLE_DIR / "source_interface_discovery.json"),
        "agent_configs": str(TABLE_DIR / "agent_config_inventory.json"),
        "output_schemas": str(TABLE_DIR / "author_output_schema_inventory.json"),
        "environment": str(MANIFEST_DIR / "environment_and_repo.json"),
        "worktree": str(MANIFEST_DIR / "coopeval_worktree.json"),
        "native_tests": str(MANIFEST_DIR / "native_target_tests.json"),
    },
    "next_notebook_scope": [
        "construct one official-engine Traveler's Dilemma state",
        "recover exact subject prompt and continuous claim parser",
        "use the same local Qwen subject checkpoint",
        "capture final-prompt-token activations",
        "run a matched text-only auditor",
        "test a small parameterized payoff sweep",
        "freeze grouped states before any causal extension",
    ],
}

handoff_path = MANIFEST_DIR / "notebook04_adapter_handoff.json"
handoff_path.write_text(json.dumps(handoff, indent=2, default=str))

print(json.dumps({
    "coopeval_commit": commit,
    "runner_help_passed": handoff["coopeval"]["runner_help_passed"],
    "targeted_native_tests_attempted": handoff["coopeval"]["targeted_native_tests_attempted"],
    "targeted_native_tests_returncode": handoff["coopeval"]["targeted_native_tests_returncode"],
    "game_config": handoff["target"]["game_config"],
    "mechanism_config": handoff["target"]["mechanism_config"],
    "game_class": handoff["target"]["game_class"],
    "mechanism_class": handoff["target"]["mechanism_class"],
}, indent=2, default=str))

print("\nNotebook 04 complete.")
print(f"Adapter handoff: {handoff_path}")
print(f"Run directory: {RUN_OUTPUT_DIR}")


{
  "coopeval_commit": "8559e858b5cc33e0a29ac592c0b21eab007feb57",
  "runner_help_passed": true,
  "targeted_native_tests_attempted": false,
  "targeted_native_tests_returncode": null,
  "game_config": "configs/games/travellers_dilemma.yaml",
  "mechanism_config": "configs/mechanisms/no_mechanism.yaml",
  "game_class": {
    "module": "coopeval.games.travellers_dilemma",
    "class": "TravellersDilemma",
    "constructor": "(*, min_claim: 'int', num_actions: 'int', claim_spacing: 'int', bonus: 'float') -> 'None'",
    "source_file": "/workspace/latent-reservations/vendor/coopeval/src/coopeval/games/travellers_dilemma.py",
    "public_methods": [
      {
        "name": "add_mediator_action",
        "signature": "(self) -> None"
      },
      {
        "name": "get_action_self_payoff",
        "signature": "(self, action: 'Action') -> 'float'"
      },
      {
        "name": "get_actions_payoff",
        "signature": "(self, actions: 'Sequence[Action]') -> 'Sequence[float]'"
      },